# Landmark Recognition using SIFT (Google Colab + Google Drive version)

**Pipeline:** database → SIFT features → BFMatcher (k=2) + Lowe's ratio test → decision with thresholds → homography + RANSAC localization → evaluation.

**Your Drive layout:** one folder with 5 landmark sub-folders (4-5 images each). Folder name = landmark label.
```
MyDrive/<your folder>/
    eiffel_tower/   taj_mahal/   colosseum/   ...
```
**Important idea (hold-out):** the last image of each landmark is *removed* from the database and used later as a test query.
So the system is tested on photos it has never seen. Run the cells from top to bottom.

## 0. Mount Google Drive
A pop-up will ask for permission. If your folder was *shared* with you, first open Drive and use
"Organize → Add shortcut to Drive" so that it appears under **My Drive**.

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not running in Colab: set DB_DIR manually in the settings cell.")

## 1. Imports and settings
All tunable numbers are here. Leave `DB_DIR = ""` to let the notebook find your folder automatically.

In [ ]:
import csv
import os
import time

import cv2
import matplotlib.pyplot as plt
import numpy as np

# ---------------- PATH SETTINGS ----------------
DB_DIR = ""                                   # "" = auto-detect. Or e.g. "/content/drive/MyDrive/Landmarks"
DRIVE_ROOT = "/content/drive/MyDrive"         # where auto-detect searches
OUT_DIR = "/content/drive/MyDrive/landmark_results"  # screenshots + csv are saved here
UNKNOWN_IMAGE_PATH = ""                       # optional: path to a non-database landmark photo (e.g. a pyramid)
EVAL_LANDMARK = ""                            # landmark used for Task 6 tests. "" = first one (alphabetical)

IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
EXCLUDE_FOLDERS = {"queries", "query", "results", "test", "tests", "output"}  # not landmarks

# ---------------- ALGORITHM SETTINGS ----------------
MAX_SIDE = 800            # shrink big images -> faster SIFT
RATIO = 0.75              # Lowe's ratio threshold
RANSAC_THRESHOLD = 5.0    # max pixel error for a match to be an inlier
HOLD_OUT_LAST = True      # remove last image of each landmark from the database (used as test)

MIN_GOOD_MATCHES = 25     # threshold 1: enough reliable matches
MIN_INLIERS = 12          # threshold 2: enough RANSAC inliers
MIN_MARGIN = 1.3          # threshold 3: best must beat runner-up by 30%

print("OpenCV version:", cv2.__version__)

## 2. Find the database folder
A valid database folder contains **at least 5 sub-folders that hold images**. This cell searches your Drive (3 levels deep).

In [ ]:
def has_images(folder):
    return any(f.lower().endswith(IMAGE_EXTENSIONS) for f in os.listdir(folder))


def list_landmark_folders(folder):
    """Sub-folders of `folder` that contain images (excluding names like 'queries')."""
    found = []
    for name in sorted(os.listdir(folder)):
        path = os.path.join(folder, name)
        if os.path.isdir(path) and name.lower() not in EXCLUDE_FOLDERS and has_images(path):
            found.append(name)
    return found


def find_database_folders(root, max_depth=3):
    """Walk through Drive and return folders that look like a landmark database."""
    candidates = []
    root_depth = root.rstrip("/").count("/")
    for current, dirs, _ in os.walk(root):
        dirs[:] = [d for d in dirs if not d.startswith(".")]      # skip hidden folders
        if current.count("/") - root_depth >= max_depth:
            dirs[:] = []                                          # do not go deeper
        if len(list_landmark_folders(current)) >= 5:
            candidates.append(current)
    return candidates


if not DB_DIR:
    candidates = find_database_folders(DRIVE_ROOT)
    print("Candidate folders found:")
    for c in candidates:
        print("  ", c, "->", list_landmark_folders(c))
    if not candidates:
        raise FileNotFoundError("No folder with 5 landmark sub-folders found. Set DB_DIR manually.")
    DB_DIR = candidates[0]

print("\nUsing DB_DIR =", DB_DIR)
print("Landmarks    =", list_landmark_folders(DB_DIR))

## 3. Image helpers
- OpenCV loads images as **BGR**, matplotlib shows **RGB**, so we convert before displaying.
- SIFT uses only intensity (gradients), so we convert to **grayscale**.

In [ ]:
def load_image(path):
    """Read an image from disk, shrink it if too large. Returns a BGR image."""
    bgr = cv2.imread(path)
    if bgr is None:
        raise FileNotFoundError(f"Cannot read image: {path}")
    h, w = bgr.shape[:2]
    scale = MAX_SIDE / max(h, w)
    if scale < 1:  # only shrink, never enlarge
        bgr = cv2.resize(bgr, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
    return bgr


def to_gray(bgr):
    """Remove color: SIFT works on intensity only."""
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)


def show(title, bgr, save_name=None):
    """Display an image inline; optionally save it to OUT_DIR."""
    if save_name:
        os.makedirs(OUT_DIR, exist_ok=True)
        cv2.imwrite(os.path.join(OUT_DIR, save_name), bgr)
    plt.figure(figsize=(10, 6))
    plt.imshow(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis("off")
    plt.show()


def draw_keypoints(bgr, keypoints):
    """Circle size = keypoint scale, line = orientation."""
    return cv2.drawKeypoints(bgr, keypoints, None,
                             flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)

## 4. SIFT feature extraction and database (Task 1, 2, 3)
**Keypoint** = a distinctive spot (corner/blob) with position, scale and orientation.
**Descriptor** = 128 numbers (4×4 cells × 8 gradient bins) describing the patch around a keypoint.

- Scale-space (Difference of Gaussians) → robust to **resizing**
- Dominant orientation → robust to **rotation**
- Normalized gradients → robust to **brightness change**

One function, `extract_sift()`, is used for both reference and query images.
`build_database()` returns **two** things: the database (used for matching) and the hold-out images (used only for testing).

In [ ]:
def extract_sift(sift, bgr):
    """Return (keypoints, descriptors). descriptors shape = (N, 128)."""
    gray = to_gray(bgr)
    keypoints, descriptors = sift.detectAndCompute(gray, None)
    return keypoints, descriptors


def make_entry(sift, label, path):
    """Load one image and extract its features. Returns a dict, or None if unusable."""
    bgr = load_image(path)
    keypoints, descriptors = extract_sift(sift, bgr)
    if descriptors is None or len(keypoints) < 2:
        print(f"  [skip] {path}: too few keypoints")
        return None
    return {"label": label, "path": path, "bgr": bgr,
            "keypoints": keypoints, "descriptors": descriptors}


def build_database(sift):
    """
    Read DB_DIR/<landmark>/<image>.
    Returns (database, heldout):
        database : list of entries used for matching
        heldout  : dict label -> entry (unseen test image), only if HOLD_OUT_LAST
    """
    database, heldout = [], {}
    for label in list_landmark_folders(DB_DIR):
        folder = os.path.join(DB_DIR, label)
        files = sorted(f for f in os.listdir(folder) if f.lower().endswith(IMAGE_EXTENSIONS))

        # keep the last image for testing (only if the landmark has at least 3 images)
        test_file = files[-1] if (HOLD_OUT_LAST and len(files) >= 3) else None

        for filename in files:
            entry = make_entry(sift, label, os.path.join(folder, filename))
            if entry is None:
                continue
            if filename == test_file:
                heldout[label] = entry
                tag = "HELD-OUT (test)"
            else:
                database.append(entry)
                tag = "database"
            print(f"  [{tag:<15}] {label:<20} {filename:<25} {len(entry['keypoints'])} keypoints")

    if not database:
        raise RuntimeError("Database is empty. Check DB_DIR and image formats.")
    return database, heldout

## 5. Matching (Task 4)
- **BFMatcher + `NORM_L2`**: compares every query descriptor with every reference descriptor (Euclidean distance).
- **KNN (k=2)**: keep the 2 nearest reference descriptors for each query descriptor.
- **Lowe's ratio test**: keep a match only if `best < 0.75 × second_best`; otherwise it is ambiguous.
- Per landmark we keep the **best** reference image (max, not sum), so a landmark with more photos is not favored.

In [ ]:
def match_with_ratio_test(bf, query_desc, ref_desc):
    """KNN (k=2) + Lowe's ratio test. queryIdx -> query, trainIdx -> reference."""
    if ref_desc is None or len(ref_desc) < 2:
        return []
    knn_pairs = bf.knnMatch(query_desc, ref_desc, k=2)
    good = []
    for pair in knn_pairs:
        if len(pair) < 2:
            continue
        best, second = pair
        if best.distance < RATIO * second.distance:
            good.append(best)
    return good


def score_landmarks(bf, query_desc, database):
    """Best reference image per landmark, ranked from best to worst."""
    best_per_landmark = {}
    for entry in database:
        good = match_with_ratio_test(bf, query_desc, entry["descriptors"])
        label = entry["label"]
        if label not in best_per_landmark or len(good) > len(best_per_landmark[label]["good"]):
            best_per_landmark[label] = {"label": label, "entry": entry, "good": good}
    return sorted(best_per_landmark.values(), key=lambda r: len(r["good"]), reverse=True)

## 6. Localization (Task 5)
- **Homography (3×3 matrix H)** maps points of a flat object from the reference image to the query image (needs ≥ 4 point pairs).
- **RANSAC** repeatedly picks 4 random matches, computes H, and counts how many matches agree (**inliers**). It keeps the best H and ignores wrong matches.
- We project the 4 reference corners with `perspectiveTransform` and draw a polygon.
- If the geometry is invalid, we return a message instead of a polygon.

In [ ]:
def localize(query_kp, query_shape, entry, good):
    """Homography + RANSAC. Returns a dict and never crashes on bad geometry."""
    result = {"ok": False, "inliers": 0, "corners": None,
              "inlier_mask": None, "message": ""}

    if len(good) < 4:
        result["message"] = "Not enough matches for homography (<4)."
        return result

    ref_pts = np.float32([entry["keypoints"][m.trainIdx].pt for m in good]).reshape(-1, 1, 2)
    qry_pts = np.float32([query_kp[m.queryIdx].pt for m in good]).reshape(-1, 1, 2)

    H, mask = cv2.findHomography(ref_pts, qry_pts, cv2.RANSAC, RANSAC_THRESHOLD)
    if H is None or mask is None:
        result["message"] = "RANSAC failed to find a homography."
        return result

    inlier_mask = mask.ravel().tolist()
    result["inliers"] = int(sum(inlier_mask))
    result["inlier_mask"] = inlier_mask

    if result["inliers"] < MIN_INLIERS:
        result["message"] = f"Too few inliers ({result['inliers']} < {MIN_INLIERS}); geometry unreliable."
        return result

    h, w = entry["bgr"].shape[:2]
    ref_corners = np.float32([[0, 0], [w, 0], [w, h], [0, h]]).reshape(-1, 1, 2)
    corners = cv2.perspectiveTransform(ref_corners, H)

    qh, qw = query_shape[:2]
    area = cv2.contourArea(corners)
    if not cv2.isContourConvex(np.int32(corners)):
        result["message"] = "Projected polygon is not convex (invalid)."
        return result
    if not (0.01 * qh * qw < area < 3.0 * qh * qw):
        result["message"] = "Projected polygon has an unreasonable size."
        return result

    result["ok"] = True
    result["corners"] = corners
    result["message"] = "Localization succeeded."
    return result

## 7. Full pipeline for one query image
We do **not** classify by the largest match count alone. **All three** checks must pass:
1. matches ≥ `MIN_GOOD_MATCHES`
2. best candidate beats the runner-up by `MIN_MARGIN`
3. RANSAC geometry is valid

In [ ]:
def recognize(query_bgr, sift, bf, database):
    """Extraction -> matching -> decision -> localization. Returns a dict."""
    start = time.perf_counter()

    query_kp, query_desc = extract_sift(sift, query_bgr)
    if query_desc is None or len(query_kp) < 2:
        return {"decision": "REJECTED", "reason": "No usable keypoints",
                "predicted": "none", "best_candidate": "none", "n_keypoints": 0,
                "n_good": 0, "inliers": 0, "time": time.perf_counter() - start,
                "query_kp": query_kp, "ranking": [], "top": None, "loc": None}

    ranking = score_landmarks(bf, query_desc, database)
    top = ranking[0]
    n_good = len(top["good"])
    runner_up = len(ranking[1]["good"]) if len(ranking) > 1 else 0

    loc = localize(query_kp, query_bgr.shape, top["entry"], top["good"])

    reasons = []
    if n_good < MIN_GOOD_MATCHES:
        reasons.append(f"good matches {n_good} < {MIN_GOOD_MATCHES}")
    if n_good < MIN_MARGIN * max(runner_up, 1):
        reasons.append(f"not clearly better than runner-up ({n_good} vs {runner_up})")
    if not loc["ok"]:
        reasons.append(loc["message"])

    recognized = len(reasons) == 0
    return {
        "decision": "RECOGNIZED" if recognized else "REJECTED",
        "reason": "; ".join(reasons) if reasons else "All checks passed",
        "predicted": top["label"] if recognized else "unknown",
        "best_candidate": top["label"],
        "n_keypoints": len(query_kp),
        "n_good": n_good,
        "inliers": loc["inliers"],
        "time": time.perf_counter() - start,
        "query_kp": query_kp,
        "ranking": ranking,
        "top": top,
        "loc": loc,
    }

## 8. Display results (Expected Output 1-7)

In [ ]:
def print_report(res):
    print("\n" + "=" * 55)
    print("RANKING (reliable matches per landmark)")
    for i, r in enumerate(res["ranking"], 1):
        print(f"  {i}. {r['label']:<20} {len(r['good'])}")
    print("-" * 55)
    print(f"Best candidate   : {res['best_candidate']}")
    print(f"Query keypoints  : {res['n_keypoints']}")
    print(f"Reliable matches : {res['n_good']}")
    print(f"RANSAC inliers   : {res['inliers']}")
    print(f"Processing time  : {res['time']:.3f} s")
    print(f"DECISION         : {res['decision']} -> {res['predicted']}")
    print(f"Reason           : {res['reason']}")
    print("=" * 55)


def show_results(query_bgr, res, name="query"):
    show("1. Query image", query_bgr, f"{name}_1_query.jpg")
    show(f"2. Query SIFT keypoints ({res['n_keypoints']})",
         draw_keypoints(query_bgr, res["query_kp"]), f"{name}_2_keypoints.jpg")

    if not res["ranking"]:
        return
    top, loc = res["top"], res["loc"]

    # If homography worked, draw only the inlier matches
    mask = loc["inlier_mask"] if loc["ok"] else None
    match_img = cv2.drawMatches(
        query_bgr, res["query_kp"], top["entry"]["bgr"], top["entry"]["keypoints"],
        top["good"], None, matchesMask=mask,
        flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
    show(f"3. Matches with {top['label']}", match_img, f"{name}_3_matches.jpg")

    located = query_bgr.copy()
    if loc["ok"]:
        cv2.polylines(located, [np.int32(loc["corners"])], True, (0, 0, 255), 4)
    color = (0, 255, 0) if res["decision"] == "RECOGNIZED" else (0, 0, 255)
    label = f"{res['decision']}: {res['predicted']} | matches={res['n_good']} | inliers={res['inliers']}"
    cv2.putText(located, label, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
    if not loc["ok"]:
        cv2.putText(located, "Localization unreliable", (10, 60),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 165, 255), 2)
    show("6. Localization result", located, f"{name}_6_localized.jpg")

## 9. Evaluation helpers (Task 6)
| Test | Query image | Expected |
|---|---|---|
| Original | a database image, unchanged | the landmark |
| Rotated / Resized / Reduced brightness | the same image, modified | the landmark |
| Different viewpoint | the **hold-out** photo of that landmark (never seen) | the landmark |
| Unknown landmark | your `UNKNOWN_IMAGE_PATH`; if empty, the hold-out photo of the *last* landmark with that landmark **removed** from the database | `unknown` |

In [ ]:
def rotate_image(bgr, angle=30):
    h, w = bgr.shape[:2]
    M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
    return cv2.warpAffine(bgr, M, (w, h))


def resize_image(bgr, factor=0.5):
    return cv2.resize(bgr, None, fx=factor, fy=factor, interpolation=cv2.INTER_AREA)


def darken_image(bgr, factor=0.4):
    return np.clip(bgr.astype(np.float32) * factor, 0, 255).astype(np.uint8)


def save_table(header, rows, filename):
    os.makedirs(OUT_DIR, exist_ok=True)
    path = os.path.join(OUT_DIR, filename)
    with open(path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(header)
        writer.writerows(rows)
    print(f"Table saved to {path}")


def build_task6_tests(database, heldout):
    """Return a list of (test name, query image, expected label, database to use)."""
    labels = sorted({e["label"] for e in database})
    eval_label = EVAL_LANDMARK or labels[0]
    base_entry = next(e for e in database if e["label"] == eval_label)
    original = base_entry["bgr"]

    tests = [
        ("Original",           original,                eval_label, database),
        ("Rotated 30 deg",     rotate_image(original),  eval_label, database),
        ("Resized 50%",        resize_image(original),  eval_label, database),
        ("Reduced brightness", darken_image(original),  eval_label, database),
    ]

    if eval_label in heldout:
        tests.append(("Different viewpoint", heldout[eval_label]["bgr"], eval_label, database))
    else:
        print("[warning] No hold-out image for", eval_label, "- viewpoint test skipped.")

    # Unknown landmark test
    if UNKNOWN_IMAGE_PATH and os.path.exists(UNKNOWN_IMAGE_PATH):
        tests.append(("Unknown landmark", load_image(UNKNOWN_IMAGE_PATH), "unknown", database))
    else:
        unknown_label = labels[-1] if labels[-1] != eval_label else labels[0]
        if unknown_label in heldout:
            smaller_db = [e for e in database if e["label"] != unknown_label]
            tests.append((f"Unknown ({unknown_label} removed from DB)",
                          heldout[unknown_label]["bgr"], "unknown", smaller_db))
        else:
            print("[warning] No unknown-landmark test available.")
    return tests


def run_evaluation(sift, bf, database, heldout, show_images=True):
    header = ["Test", "Predicted", "Keypoints", "Good matches",
              "Inliers", "Decision", "Time (s)", "Correct?"]
    rows = []
    for name, image, expected, db in build_task6_tests(database, heldout):
        res = recognize(image, sift, bf, db)
        correct = "YES" if res["predicted"] == expected else "NO"
        rows.append([name, res["predicted"], res["n_keypoints"], res["n_good"],
                     res["inliers"], res["decision"], f"{res['time']:.3f}", correct])
        if show_images:
            safe = "".join(c if c.isalnum() else "_" for c in name)
            show_results(image, res, name=safe)
    save_table(header, rows, "evaluation.csv")
    return header, rows

## 10. RUN: build the database
Shows which images go to the database and which are held out. Then displays a few reference images with their SIFT keypoints (Task 2, step 4).

In [ ]:
sift = cv2.SIFT_create()          # SIFT detector + descriptor
bf = cv2.BFMatcher(cv2.NORM_L2)   # brute force + Euclidean distance

print("Building database...")
database, heldout = build_database(sift)
print(f"\nDatabase images: {len(database)} | Hold-out test images: {len(heldout)}")

shown = set()
for entry in database:                      # one reference image per landmark
    if entry["label"] in shown:
        continue
    shown.add(entry["label"])
    show(f"Reference: {entry['label']} ({len(entry['keypoints'])} keypoints)",
         draw_keypoints(entry["bgr"], entry["keypoints"]),
         f"reference_{entry['label']}.jpg")

## 11. RUN: recognize one query image
Leave `QUERY_PATH` empty to use the hold-out photo of the first landmark, or paste any image path from Drive.

In [ ]:
QUERY_PATH = ""   # e.g. "/content/drive/MyDrive/test_photos/my_photo.jpg"

if QUERY_PATH:
    query = load_image(QUERY_PATH)
else:
    first_label = sorted(heldout)[0]
    query = heldout[first_label]["bgr"]
    print("Using hold-out image of:", first_label)

res = recognize(query, sift, bf, database)
print_report(res)
show_results(query, res, name="single_query")

## 12. RUN: Task 6 performance table
Copy this table into your report and take screenshots of the images above.

In [ ]:
header, rows = run_evaluation(sift, bf, database, heldout, show_images=True)

print("\n" + " | ".join(header))
for row in rows:
    print(" | ".join(str(x) for x in row))

## 13. Bonus: test every hold-out image
One unseen photo per landmark, no plots. This gives a quick accuracy number for your discussion section.

In [ ]:
header2 = ["True landmark", "Predicted", "Good matches", "Inliers", "Decision", "Time (s)", "Correct?"]
rows2 = []
for label, entry in sorted(heldout.items()):
    res = recognize(entry["bgr"], sift, bf, database)
    correct = "YES" if res["predicted"] == label else "NO"
    rows2.append([label, res["predicted"], res["n_good"], res["inliers"],
                  res["decision"], f"{res['time']:.3f}", correct])

print(" | ".join(header2))
for row in rows2:
    print(" | ".join(str(x) for x in row))

n_correct = sum(r[-1] == "YES" for r in rows2)
print(f"\nAccuracy on unseen photos: {n_correct}/{len(rows2)}")
save_table(header2, rows2, "heldout_accuracy.csv")

## 14. Discussion (for your report)
**Strengths:** handles rotation, scale and brightness changes; needs no training; the polygon gives an explainable result.

**Limitations:**
- Brute-force matching gets slow as the database grows (use FLANN or a vocabulary tree).
- Big viewpoint changes break the planar-homography assumption (landmarks are 3D).
- Repeating textures (e.g. Colosseum arches) create ambiguous matches.
- Night photos, occlusion, and crowds reduce matches.
- Thresholds are hand-tuned and may not generalize; the test set is very small.

**Improvements:** more reference images, verify the top 3 candidates with RANSAC, RootSIFT, or deep features (SuperPoint + LightGlue).

**Debugging tips:**
- `No folder with 5 landmark sub-folders found`: the folder is not in My Drive. Add a shortcut, or set `DB_DIR` manually.
- Everything is REJECTED: read the `reason` line, then lower `MIN_GOOD_MATCHES` (try 15).
- An unknown image is wrongly RECOGNIZED: raise `MIN_GOOD_MATCHES` or lower `RATIO` to 0.7.
- The Great Wall gets few matches: it has little unique texture. Add a close-up image to the database.